# Задача на оптимизацию

In [64]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [65]:
# матрица для оптимизации
import pandas as pd

X = pd.read_csv('/content/drive/MyDrive/data/X_aim.csv', header=None, index_col=False).values
X.shape

(50, 100)

In [66]:
X[0][0]

np.float64(0.263)

In [67]:
# функция для оптимизации
import numpy as np
from numba import jit

@jit(nopython=True)
def func4opt(I):
    global X
    assert len(set(I)) == 100
    z = X[:, I[0]] * len(I)
    for i in I[1:]:
        z += np.sin(10 * X[:, i] + z)
    return np.sum(z ** 2)

In [68]:
# значение функции на списке [0,1,...,99]
I = list(range(100))
func4opt(I)

365774.24580511125

In [69]:
# значение функции на случайном списке
np.random.shuffle(I)
func4opt(I)

165228.81411478072

In [70]:
from typing import List, Tuple
import random
import math

def get_neighbor(solution):
    """Улучшенная генерация соседей с разными типами мутаций"""
    new_solution = solution.copy()
    n = len(new_solution)

    # Случайный выбор типа мутации
    mutation_type = random.choice(['swap', 'insert', 'invert', 'scramble'])

    if mutation_type == 'swap':
        # Классическая перестановка двух элементов
        idx1, idx2 = random.sample(range(n), 2)
        new_solution[idx1], new_solution[idx2] = new_solution[idx2], new_solution[idx1]

    elif mutation_type == 'insert':
        # Вставка элемента на новую позицию
        idx = random.randint(0, n-2)
        new_solution.insert(random.randint(0, n-1), new_solution.pop(idx))

    elif mutation_type == 'invert':
        # Инверсия подпоследовательности
        start, end = sorted(random.sample(range(n), 2))
        new_solution[start:end+1] = new_solution[start:end+1][::-1]

    elif mutation_type == 'scramble':
        # Перемешивание подпоследовательности
        start, end = sorted(random.sample(range(n), 2))
        random.shuffle(new_solution[start:end+1])

    return new_solution

def adaptive_cooling(T, alpha, improvement_ratio):
    """Адаптивное охлаждение с учетом прогресса"""
    if improvement_ratio > 0.1:  # Хороший прогресс - охлаждаем медленнее
        return T * (alpha ** 0.5)
    elif improvement_ratio < 0.01:  # Плохой прогресс - охлаждаем быстрее
        return T * (alpha ** 2)
    else:
        return T * alpha

def multi_start_sa(func, params, n_starts=5):
    best_global = None
    best_score = float('inf')

    for i in range(n_starts):
        print(f"\nЗапуск {i+1}/{n_starts}")
        initial = list(range(100))
        random.shuffle(initial)

        solution, score = improved_simulated_annealing(
            func,
            initial,
            T_start=params['T_start'],
            T_end=params['T_end'],
            alpha=params['alpha'],
            max_iter_at_temp=params['max_iter_at_temp']
        )

        # Применяем локальный поиск к лучшему решению
        solution, score = local_search(func, solution, score)

        if score < best_score:
            best_global = solution
            best_score = score
            print(f"Новый глобальный минимум: {best_score}")

    return best_global, best_score


def local_search(func, solution, initial_score):
    """Жадный локальный поиск с операцией swap"""
    current = solution.copy()
    current_score = initial_score
    improved = True

    while improved:
        improved = False
        for i in range(len(current)):
            for j in range(i+1, len(current)):
                # Пробуем swap
                current[i], current[j] = current[j], current[i]
                new_score = func(current)

                if new_score < current_score:
                    current_score = new_score
                    improved = True
                else:
                    # Отменяем изменение
                    current[i], current[j] = current[j], current[i]

    return current, current_score

# --- Алгоритм симуляции отжига ---
def improved_simulated_annealing(func, initial_solution, T_start, T_end, alpha, max_iter_at_temp):
    current = initial_solution.copy()
    current_score = func(current)

    best = current.copy()
    best_score = current_score

    T = T_start
    iteration = 0
    last_improvement = 0
    improvement_history = []

    while T > T_end and iteration < 100000:  # Добавляем ограничение по итерациям
        accepted = 0
        improved = 0

        for _ in range(max_iter_at_temp):
            # Генерация соседа
            neighbor = get_neighbor(current)
            neighbor_score = func(neighbor)

            delta = neighbor_score - current_score

            # Критерий принятия
            if delta < 0 or random.random() < math.exp(-delta / T):
                current, current_score = neighbor, neighbor_score
                accepted += 1

                if delta < 0:
                    improved += 1

            # Обновление лучшего решения
            if current_score < best_score:
                best, best_score = current, current_score
                last_improvement = iteration

        # Адаптивное охлаждение
        improvement_ratio = improved / max_iter_at_temp
        improvement_history.append(improvement_ratio)
        if len(improvement_history) > 10:
            improvement_history.pop(0)

        avg_improvement = sum(improvement_history) / len(improvement_history)
        T = adaptive_cooling(T, alpha, avg_improvement)

        # Ранняя остановка при отсутствии улучшений
        if iteration - last_improvement > 5000:
            print(f"Ранняя остановка на итерации {iteration}")
            break

        # Логирование
        if iteration % 100 == 0:
            print(f"Iter {iteration}: T={T:.2f}, Current={current_score:.2f}, Best={best_score:.2f}, Accept={accepted/max_iter_at_temp:.2f}, Improve={improved/max_iter_at_temp:.2f}")

        iteration += 1

    return best, best_score

params = {
    'T_start': 100,    # Более низкая начальная температура
    'T_end': 1e-6,
    'alpha': 0.9995,       # Более быстрое охлаждение
    'max_iter_at_temp': 500,
    'n_starts': 3        # Количество многостартовых запусков
}

# Запуск алгоритма с параметрами
final_solution, final_score = multi_start_sa(func4opt, params, n_starts=params['n_starts'])
print(f"\nЛучшее найденное решение: {final_score}")
print(f"Перестановка: {final_solution}")

# Создание submission файла
submission = pd.DataFrame({
    'row_id': range(100),
    'idx': final_solution
})
submission.to_csv('/content/drive/MyDrive/data/submission.csv', index=False)


Запуск 1/3
Iter 0: T=99.95, Current=121313.23, Best=121313.23, Accept=0.33, Improve=0.06
Iter 100: T=93.10, Current=1448.79, Best=973.36, Accept=0.37, Improve=0.07
Iter 200: T=88.56, Current=1085.00, Best=708.41, Accept=0.37, Improve=0.06
Iter 300: T=84.24, Current=1918.11, Best=607.90, Accept=0.40, Improve=0.05
Iter 400: T=80.13, Current=1212.17, Best=607.90, Accept=0.39, Improve=0.05
Iter 500: T=76.22, Current=1631.77, Best=607.90, Accept=0.38, Improve=0.06
Iter 600: T=72.50, Current=1679.68, Best=607.90, Accept=0.41, Improve=0.05
Iter 700: T=68.96, Current=1381.31, Best=607.90, Accept=0.30, Improve=0.03
Iter 800: T=65.60, Current=1597.91, Best=607.90, Accept=0.32, Improve=0.05
Iter 900: T=62.40, Current=1448.58, Best=607.90, Accept=0.33, Improve=0.03
Iter 1000: T=59.36, Current=1180.07, Best=598.31, Accept=0.34, Improve=0.05
Iter 1100: T=56.46, Current=1164.49, Best=598.31, Accept=0.26, Improve=0.01
Iter 1200: T=53.71, Current=1146.41, Best=598.31, Accept=0.35, Improve=0.03
Iter 13